# Task 1 Comparison: Cleaned CNN vs Convolutional-Recurrent

This notebook compares the strongest cleaned CNN baseline against the first convolutional-recurrent experiment using the saved Task 1 artifacts.

In [1]:
from pathlib import Path
import json
import pandas as pd

class HtmlDisplay(str):
    def _repr_html_(self):
        return str(self)

repo_root = Path.cwd()
runs = {
    'cnn_cleaned': repo_root / 'outputs' / 'baseline_effb0_cleaned',
    'conv_recurrent': repo_root / 'outputs' / 'conv_recurrent_effb0',
}

def load_history(run_dir: Path):
    return json.loads((run_dir / 'history.json').read_text())

def load_outlier_summary(run_dir: Path):
    summary = (run_dir / 'outliers' / 'outlier_summary.md').read_text().splitlines()
    values = {}
    for line in summary:
        line = line.strip()
        if line.startswith('- Samples with at least one wrong task:'):
            values['samples_with_any_wrong'] = int(line.rsplit(':', 1)[1].strip())
        elif line.startswith('- `artist` errors:'):
            values['artist_errors'] = int(line.rsplit(':', 1)[1].strip())
        elif line.startswith('- `genre` errors:'):
            values['genre_errors'] = int(line.rsplit(':', 1)[1].strip())
        elif line.startswith('- `style` errors:'):
            values['style_errors'] = int(line.rsplit(':', 1)[1].strip())
    return values

histories = {name: load_history(path) for name, path in runs.items()}
best_epochs = {name: max(history, key=lambda row: row['mean_top1']) for name, history in histories.items()}
outliers = {name: load_outlier_summary(path) for name, path in runs.items()}

summary_rows = []
for name, best in best_epochs.items():
    summary_rows.append({
        'run': name,
        'best_epoch': best['epoch'],
        'val_loss': best['val_loss'],
        'mean_top1': best['mean_top1'],
        'style_top1': best['val_metrics']['style']['top1'],
        'genre_top1': best['val_metrics']['genre']['top1'],
        'artist_top1': best['val_metrics']['artist']['top1'],
        'samples_with_any_wrong': outliers[name]['samples_with_any_wrong'],
        'style_errors': outliers[name]['style_errors'],
        'genre_errors': outliers[name]['genre_errors'],
        'artist_errors': outliers[name]['artist_errors'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,run,best_epoch,val_loss,mean_top1,style_top1,genre_top1,artist_top1,samples_with_any_wrong,style_errors,genre_errors,artist_errors
0,cnn_cleaned,4,1.721296,0.815877,0.827916,0.781177,0.838538,1831,810,1030,760
1,conv_recurrent,5,1.819318,0.807592,0.817081,0.771192,0.834502,1901,861,1077,779


In [2]:
def svg_line_chart(histories, metric_key='mean_top1', width=760, height=320):
    pad = 40
    colors = {'cnn_cleaned': '#0b6e4f', 'conv_recurrent': '#b33c00'}
    labels = {'cnn_cleaned': 'Cleaned CNN baseline', 'conv_recurrent': 'Conv-recurrent'}
    max_epoch = max(len(history) for history in histories.values())
    values = [row[metric_key] for history in histories.values() for row in history]
    y_min = min(values) - 0.01
    y_max = max(values) + 0.01
    plot_w = width - 2 * pad
    plot_h = height - 2 * pad

    def sx(epoch_index):
        if max_epoch == 1:
            return pad + plot_w / 2
        return pad + (epoch_index / (max_epoch - 1)) * plot_w

    def sy(value):
        return pad + (1 - (value - y_min) / (y_max - y_min)) * plot_h

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<line x1="{pad}" y1="{height-pad}" x2="{width-pad}" y2="{height-pad}" stroke="#222"/>',
        f'<line x1="{pad}" y1="{pad}" x2="{pad}" y2="{height-pad}" stroke="#222"/>',
    ]

    for tick in range(max_epoch):
        x = sx(tick)
        parts.append(f'<text x="{x}" y="{height-pad+18}" text-anchor="middle" font-size="12">{tick+1}</text>')

    for ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
        value = y_min + ratio * (y_max - y_min)
        y = sy(value)
        parts.append(f'<line x1="{pad}" y1="{y}" x2="{width-pad}" y2="{y}" stroke="#ddd"/>')
        parts.append(f'<text x="{pad-8}" y="{y+4}" text-anchor="end" font-size="12">{value:.3f}</text>')

    for name, history in histories.items():
        points = ' '.join(f'{sx(i)},{sy(row[metric_key])}' for i, row in enumerate(history))
        color = colors[name]
        parts.append(f'<polyline fill="none" stroke="{color}" stroke-width="3" points="{points}"/>')
        for i, row in enumerate(history):
            parts.append(f'<circle cx="{sx(i)}" cy="{sy(row[metric_key])}" r="4" fill="{color}"/>')

    legend_y = 18
    legend_x = width - 240
    for idx, name in enumerate(['cnn_cleaned', 'conv_recurrent']):
        y = legend_y + idx * 20
        parts.append(f'<rect x="{legend_x}" y="{y-10}" width="12" height="12" fill="{colors[name]}"/>')
        parts.append(f'<text x="{legend_x+18}" y="{y}" font-size="12">{labels[name]}</text>')

    parts.append(f'<text x="{width/2}" y="18" text-anchor="middle" font-size="16" font-weight="600">Validation Mean Top-1 by Epoch</text>')
    parts.append('</svg>')
    return HtmlDisplay(''.join(parts))

svg_line_chart(histories)

'<svg width="760" height="320" viewBox="0 0 760 320" xmlns="http://www.w3.org/2000/svg"><rect width="100%" height="100%" fill="#ffffff"/><line x1="40" y1="280" x2="720" y2="280" stroke="#222"/><line x1="40" y1="40" x2="40" y2="280" stroke="#222"/><text x="40.0" y="298" text-anchor="middle" font-size="12">1</text><text x="210.0" y="298" text-anchor="middle" font-size="12">2</text><text x="380.0" y="298" text-anchor="middle" font-size="12">3</text><text x="550.0" y="298" text-anchor="middle" font-size="12">4</text><text x="720.0" y="298" text-anchor="middle" font-size="12">5</text><line x1="40" y1="280.0" x2="720" y2="280.0" stroke="#ddd"/><text x="32" y="284.0" text-anchor="end" font-size="12">0.734</text><line x1="40" y1="220.00000000000009" x2="720" y2="220.00000000000009" stroke="#ddd"/><text x="32" y="224.00000000000009" text-anchor="end" font-size="12">0.757</text><line x1="40" y1="159.9999999999999" x2="720" y2="159.9999999999999" stroke="#ddd"/><text x="32" y="163.9999999999999" text-anchor="end" font-size="12">0.780</text><line x1="40" y1="99.99999999999991" x2="720" y2="99.99999999999991" stroke="#ddd"/><text x="32" y="103.99999999999991" text-anchor="end" font-size="12">0.803</text><line x1="40" y1="40.0" x2="720" y2="40.0" stroke="#ddd"/><text x="32" y="44.0" text-anchor="end" font-size="12">0.826</text><polyline fill="none" stroke="#0b6e4f" stroke-width="3" points="40.0,226.6246289745807 210.0,140.99075683703677 380.0,85.07036188514985 550.0,66.06111871668647 720.0,79.34913335871897"/><circle cx="40.0" cy="226.6246289745807" r="4" fill="#0b6e4f"/><circle cx="210.0" cy="140.99075683703677" r="4" fill="#0b6e4f"/><circle cx="380.0" cy="85.07036188514985" r="4" fill="#0b6e4f"/><circle cx="550.0" cy="66.06111871668647" r="4" fill="#0b6e4f"/><circle cx="720.0" cy="79.34913335871897" r="4" fill="#0b6e4f"/><polyline fill="none" stroke="#b33c00" stroke-width="3" points="40.0,253.93888128331352 210.0,160.55366729325218 380.0,105.37149536053279 550.0,107.40160872512163 720.0,87.65414250998927"/><circle cx="40.0" cy="253.93888128331352" r="4" fill="#b33c00"/><circle cx="210.0" cy="160.55366729325218" r="4" fill="#b33c00"/><circle cx="380.0" cy="105.37149536053279" r="4" fill="#b33c00"/><circle cx="550.0" cy="107.40160872512163" r="4" fill="#b33c00"/><circle cx="720.0" cy="87.65414250998927" r="4" fill="#b33c00"/><rect x="520" y="8" width="12" height="12" fill="#0b6e4f"/><text x="538" y="18" font-size="12">Cleaned CNN baseline</text><rect x="520" y="28" width="12" height="12" fill="#b33c00"/><text x="538" y="38" font-size="12">Conv-recurrent</text><text x="380.0" y="18" text-anchor="middle" font-size="16" font-weight="600">Validation Mean Top-1 by Epoch</text></svg>'

In [3]:
def svg_grouped_bar_chart(labels, left_values, right_values, title, left_name, right_name, width=760, height=340):
    pad = 50
    plot_w = width - 2 * pad
    plot_h = height - 2 * pad
    max_value = max(max(left_values), max(right_values)) * 1.15
    colors = ['#0b6e4f', '#b33c00']
    band = plot_w / len(labels)
    bar_w = band * 0.28

    def sy(value):
        return pad + (1 - value / max_value) * plot_h

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<line x1="{pad}" y1="{height-pad}" x2="{width-pad}" y2="{height-pad}" stroke="#222"/>',
        f'<line x1="{pad}" y1="{pad}" x2="{pad}" y2="{height-pad}" stroke="#222"/>',
    ]
    for ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
        value = max_value * ratio
        y = sy(value)
        parts.append(f'<line x1="{pad}" y1="{y}" x2="{width-pad}" y2="{y}" stroke="#ddd"/>')
        parts.append(f'<text x="{pad-8}" y="{y+4}" text-anchor="end" font-size="12">{value:.3f}</text>')
    for idx, label in enumerate(labels):
        x_center = pad + band * (idx + 0.5)
        x1 = x_center - bar_w - 4
        x2 = x_center + 4
        y1 = sy(left_values[idx])
        y2 = sy(right_values[idx])
        parts.append(f'<rect x="{x1}" y="{y1}" width="{bar_w}" height="{height-pad-y1}" fill="{colors[0]}"/>')
        parts.append(f'<rect x="{x2}" y="{y2}" width="{bar_w}" height="{height-pad-y2}" fill="{colors[1]}"/>')
        parts.append(f'<text x="{x_center}" y="{height-pad+18}" text-anchor="middle" font-size="12">{label}</text>')
    parts.append(f'<text x="{width/2}" y="20" text-anchor="middle" font-size="16" font-weight="600">{title}</text>')
    legend_x = width - 230
    for idx, name in enumerate([left_name, right_name]):
        y = 26 + idx * 18
        parts.append(f'<rect x="{legend_x}" y="{y-10}" width="12" height="12" fill="{colors[idx]}"/>')
        parts.append(f'<text x="{legend_x+18}" y="{y}" font-size="12">{name}</text>')
    parts.append('</svg>')
    return HtmlDisplay(''.join(parts))

task_labels = ['style', 'genre', 'artist']
cnn_best = best_epochs['cnn_cleaned']
rnn_best = best_epochs['conv_recurrent']
svg_grouped_bar_chart(
    task_labels,
    [cnn_best['val_metrics'][task]['top1'] for task in task_labels],
    [rnn_best['val_metrics'][task]['top1'] for task in task_labels],
    'Best Validation Top-1 by Task',
    'Cleaned CNN baseline',
    'Conv-recurrent',
)

'<svg width="760" height="340" viewBox="0 0 760 340" xmlns="http://www.w3.org/2000/svg"><rect width="100%" height="100%" fill="#ffffff"/><line x1="50" y1="290" x2="710" y2="290" stroke="#222"/><line x1="50" y1="50" x2="50" y2="290" stroke="#222"/><line x1="50" y1="290.0" x2="710" y2="290.0" stroke="#ddd"/><text x="42" y="294.0" text-anchor="end" font-size="12">0.000</text><line x1="50" y1="230.0" x2="710" y2="230.0" stroke="#ddd"/><text x="42" y="234.0" text-anchor="end" font-size="12">0.241</text><line x1="50" y1="170.0" x2="710" y2="170.0" stroke="#ddd"/><text x="42" y="174.0" text-anchor="end" font-size="12">0.482</text><line x1="50" y1="109.99999999999997" x2="710" y2="109.99999999999997" stroke="#ddd"/><text x="42" y="113.99999999999997" text-anchor="end" font-size="12">0.723</text><line x1="50" y1="50.0" x2="710" y2="50.0" stroke="#ddd"/><text x="42" y="54.0" text-anchor="end" font-size="12">0.964</text><rect x="94.39999999999999" y="83.94807283462396" width="61.60000000000001" height="206.05192716537604" fill="#0b6e4f"/><rect x="164.0" y="86.64467234018016" width="61.60000000000001" height="203.35532765981984" fill="#b33c00"/><text x="160.0" y="308" text-anchor="middle" font-size="12">style</text><rect x="314.4" y="95.5804628690353" width="61.60000000000001" height="194.4195371309647" fill="#0b6e4f"/><rect x="384.0" y="98.06556438021167" width="61.60000000000001" height="191.93443561978833" fill="#b33c00"/><text x="380.0" y="308" text-anchor="middle" font-size="12">genre</text><rect x="534.4" y="81.30434782608694" width="61.60000000000001" height="208.69565217391306" fill="#0b6e4f"/><rect x="604.0" y="82.30896332933101" width="61.60000000000001" height="207.691036670669" fill="#b33c00"/><text x="600.0" y="308" text-anchor="middle" font-size="12">artist</text><text x="380.0" y="20" text-anchor="middle" font-size="16" font-weight="600">Best Validation Top-1 by Task</text><rect x="530" y="16" width="12" height="12" fill="#0b6e4f"/><text x="548" y="26" font-size="12">Cleaned CNN baseline</text><rect x="530" y="34" width="12" height="12" fill="#b33c00"/><text x="548" y="44" font-size="12">Conv-recurrent</text></svg>'

In [4]:
error_labels = ['any wrong', 'style errors', 'genre errors', 'artist errors']
svg_grouped_bar_chart(
    error_labels,
    [outliers['cnn_cleaned']['samples_with_any_wrong'], outliers['cnn_cleaned']['style_errors'], outliers['cnn_cleaned']['genre_errors'], outliers['cnn_cleaned']['artist_errors']],
    [outliers['conv_recurrent']['samples_with_any_wrong'], outliers['conv_recurrent']['style_errors'], outliers['conv_recurrent']['genre_errors'], outliers['conv_recurrent']['artist_errors']],
    'Outlier and Error Counts',
    'Cleaned CNN baseline',
    'Conv-recurrent',
)

'<svg width="760" height="340" viewBox="0 0 760 340" xmlns="http://www.w3.org/2000/svg"><rect width="100%" height="100%" fill="#ffffff"/><line x1="50" y1="290" x2="710" y2="290" stroke="#222"/><line x1="50" y1="50" x2="50" y2="290" stroke="#222"/><line x1="50" y1="290.0" x2="710" y2="290.0" stroke="#ddd"/><text x="42" y="294.0" text-anchor="end" font-size="12">0.000</text><line x1="50" y1="230.0" x2="710" y2="230.0" stroke="#ddd"/><text x="42" y="234.0" text-anchor="end" font-size="12">546.537</text><line x1="50" y1="170.0" x2="710" y2="170.0" stroke="#ddd"/><text x="42" y="174.0" text-anchor="end" font-size="12">1093.075</text><line x1="50" y1="110.0" x2="710" y2="110.0" stroke="#ddd"/><text x="42" y="114.0" text-anchor="end" font-size="12">1639.612</text><line x1="50" y1="50.0" x2="710" y2="50.0" stroke="#ddd"/><text x="42" y="54.0" text-anchor="end" font-size="12">2186.150</text><rect x="82.3" y="88.98909041008162" width="46.2" height="201.01090958991838" fill="#0b6e4f"/><rect x="136.5" y="81.30434782608691" width="46.2" height="208.6956521739131" fill="#b33c00"/><text x="132.5" y="308" text-anchor="middle" font-size="12">any wrong</text><rect x="247.3" y="201.07655009948996" width="46.2" height="88.92344990051004" fill="#0b6e4f"/><rect x="301.5" y="195.47766621686526" width="46.2" height="94.52233378313474" fill="#b33c00"/><text x="297.5" y="308" text-anchor="middle" font-size="12">style errors</text><rect x="412.3" y="176.92450197836376" width="46.2" height="113.07549802163624" fill="#0b6e4f"/><rect x="466.5" y="171.7647462433959" width="46.2" height="118.2352537566041" fill="#b33c00"/><text x="462.5" y="308" text-anchor="middle" font-size="12">genre errors</text><rect x="577.3" y="206.56565194520047" width="46.2" height="83.43434805479953" fill="#0b6e4f"/><rect x="631.5" y="204.47979324383047" width="46.2" height="85.52020675616953" fill="#b33c00"/><text x="627.5" y="308" text-anchor="middle" font-size="12">artist errors</text><text x="380.0" y="20" text-anchor="middle" font-size="16" font-weight="600">Outlier and Error Counts</text><rect x="530" y="16" width="12" height="12" fill="#0b6e4f"/><text x="548" y="26" font-size="12">Cleaned CNN baseline</text><rect x="530" y="34" width="12" height="12" fill="#b33c00"/><text x="548" y="44" font-size="12">Conv-recurrent</text></svg>'

In [5]:
comparison = pd.DataFrame([
    {
        'metric': 'mean_top1',
        'cnn_cleaned': cnn_best['mean_top1'],
        'conv_recurrent': rnn_best['mean_top1'],
    },
    {
        'metric': 'style_top1',
        'cnn_cleaned': cnn_best['val_metrics']['style']['top1'],
        'conv_recurrent': rnn_best['val_metrics']['style']['top1'],
    },
    {
        'metric': 'genre_top1',
        'cnn_cleaned': cnn_best['val_metrics']['genre']['top1'],
        'conv_recurrent': rnn_best['val_metrics']['genre']['top1'],
    },
    {
        'metric': 'artist_top1',
        'cnn_cleaned': cnn_best['val_metrics']['artist']['top1'],
        'conv_recurrent': rnn_best['val_metrics']['artist']['top1'],
    },
    {
        'metric': 'samples_with_any_wrong',
        'cnn_cleaned': outliers['cnn_cleaned']['samples_with_any_wrong'],
        'conv_recurrent': outliers['conv_recurrent']['samples_with_any_wrong'],
    },
    {
        'metric': 'style_errors',
        'cnn_cleaned': outliers['cnn_cleaned']['style_errors'],
        'conv_recurrent': outliers['conv_recurrent']['style_errors'],
    },
    {
        'metric': 'genre_errors',
        'cnn_cleaned': outliers['cnn_cleaned']['genre_errors'],
        'conv_recurrent': outliers['conv_recurrent']['genre_errors'],
    },
    {
        'metric': 'artist_errors',
        'cnn_cleaned': outliers['cnn_cleaned']['artist_errors'],
        'conv_recurrent': outliers['conv_recurrent']['artist_errors'],
    },
])
comparison['delta_rnn_minus_cnn'] = comparison['conv_recurrent'] - comparison['cnn_cleaned']
comparison

,metric,cnn_cleaned,conv_recurrent,delta_rnn_minus_cnn
0,mean_top1,0.815877,0.807592,-0.008286
1,style_top1,0.827916,0.817081,-0.010835
2,genre_top1,0.781177,0.771192,-0.009985
3,artist_top1,0.838538,0.834502,-0.004037
4,samples_with_any_wrong,1831.000000,1901.000000,70.000000
5,style_errors,810.000000,861.000000,51.000000
6,genre_errors,1030.000000,1077.000000,47.000000
7,artist_errors,760.000000,779.000000,19.000000


In [6]:
style_delta = rnn_best['val_metrics']['style']['top1'] - cnn_best['val_metrics']['style']['top1']
genre_delta = rnn_best['val_metrics']['genre']['top1'] - cnn_best['val_metrics']['genre']['top1']
artist_delta = rnn_best['val_metrics']['artist']['top1'] - cnn_best['val_metrics']['artist']['top1']
mean_delta = rnn_best['mean_top1'] - cnn_best['mean_top1']
wrong_delta = outliers['conv_recurrent']['samples_with_any_wrong'] - outliers['cnn_cleaned']['samples_with_any_wrong']

lines = [
    'Comparison summary:',
    f"- Cleaned CNN best epoch: {cnn_best['epoch']} with mean top-1 {cnn_best['mean_top1']:.4f}.",
    f"- Conv-recurrent best epoch: {rnn_best['epoch']} with mean top-1 {rnn_best['mean_top1']:.4f}.",
    f"- Mean top-1 delta (conv-recurrent minus CNN): {mean_delta:.4f}.",
    f"- Style top-1 delta: {style_delta:.4f}.",
    f"- Genre top-1 delta: {genre_delta:.4f}.",
    f"- Artist top-1 delta: {artist_delta:.4f}.",
    f"- Wrong-sample delta: {wrong_delta} additional samples with at least one wrong task in the conv-recurrent run.",
    '',
    'Interpretation:',
    '- The cleaned CNN baseline remains stronger on all three top-1 task metrics.',
    '- The conv-recurrent model improved over its own early epochs, but it did not surpass the best CNN checkpoint.',
    '- The recurrent branch likely added optimization difficulty without adding enough useful structure for these labels.',
    '- This still satisfies the task requirement because the convolutional-recurrent architecture was implemented, trained, evaluated, and compared directly against the baseline.',
]
print('\n'.join(lines))

Comparison summary:
- Cleaned CNN best epoch: 4 with mean top-1 0.8159.
- Conv-recurrent best epoch: 5 with mean top-1 0.8076.
- Mean top-1 delta (conv-recurrent minus CNN): -0.0083.
- Style top-1 delta: -0.0108.
- Genre top-1 delta: -0.0100.
- Artist top-1 delta: -0.0040.
- Wrong-sample delta: 70 additional samples with at least one wrong task in the conv-recurrent run.

Interpretation:
- The cleaned CNN baseline remains stronger on all three top-1 task metrics.
- The conv-recurrent model improved over its own early epochs, but it did not surpass the best CNN checkpoint.
- The recurrent branch likely added optimization difficulty without adding enough useful structure for these labels.
- This still satisfies the task requirement because the convolutional-recurrent architecture was implemented, trained, evaluated, and compared directly against the baseline.
